<a href="https://colab.research.google.com/github/husayn4/hate-speech-detection-research/blob/main/HateSpeech_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# HATE SPEECH DETECTION PROJECT - DATA PREPROCESSING PIPELINE
# Stage 1: Loading, Cleaning and Category Harmonisation
# ============================================================
# This notebook loads three benchmark hate speech datasets,
# standardises them to a binary hate / non-hate classification,
# and harmonises their differing category schemes into three
# common categories: race, religion and gender.
# ============================================================

# HateXplain requires an earlier datasets library version as its
# loading script is not supported by current versions.
!pip install -q datasets==2.14.0 fsspec==2023.6.0 huggingface_hub==0.16.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.2/492.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.16.4 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.16.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.16.4 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggin

In [ ]:
import pandas as pd
import numpy as np
import re
from datasets import load_dataset

pd.set_option('display.max_colwidth', 80)
print("Libraries imported successfully")


Libraries imported successfully


In [ ]:
# ============================================================
# Text cleaning function applied consistently across all datasets
# to ensure preprocessing does not introduce cross-dataset confounds
# ============================================================

def clean_text(text):
    """Standardised text cleaning for all datasets."""
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()                          # lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text) # remove URLs
    text = re.sub(r'@\w+', '', text)             # remove @mentions
    text = re.sub(r'#', '', text)                # strip hashtag symbol, keep word
    text = re.sub(r'\s+', ' ', text)             # collapse whitespace
    text = text.strip()
    return text

# quick test
print(clean_text("Check this out @user https://example.com #HateSpeech IS BAD"))

check this out hatespeech is bad


In [ ]:
# ============================================================
# HATEXPLAIN - Load and harmonise to binary + category labels
# ============================================================
# Label mapping: HateXplain uses majority vote across 3 annotators
#   0 = hate speech  -> binary 1 (hate)
#   1 = normal       -> binary 0 (non-hate)
#   2 = offensive    -> binary 0 (non-hate, deliberately excluded
#                       from the positive class as the project
#                       focuses specifically on hate speech)
# ============================================================

print("Loading HateXplain...")
hx = load_dataset("Hate-speech-CNERG/hatexplain")
hx_train = pd.DataFrame(hx['train'])
print(f"Loaded {len(hx_train):,} training posts\n")

# --- Majority label across the three annotators ---
def hx_majority_binary(annotators):
    labels = annotators['label']           # list of 3 labels
    majority = max(set(labels), key=labels.count)
    # 0 = hate in HateXplain's scheme -> 1 in our binary scheme
    return 1 if majority == 0 else 0

# --- Map target community to our three categories ---
race_terms = {'African', 'Hispanic', 'Asian', 'Arab', 'Caucasian',
              'Indian', 'Indigenous', 'Asian'}
religion_terms = {'Islam', 'Jewish', 'Christian', 'Hindu', 'Buddhism'}
gender_terms = {'Women', 'Men'}

def hx_category(annotators):
    # collect all targets named by annotators
    targets = []
    for t_list in annotators['target']:
        targets.extend(t_list)
    # assign the first matching category found
    for t in targets:
        if t in race_terms:
            return 'race'
    for t in targets:
        if t in religion_terms:
            return 'religion'
    for t in targets:
        if t in gender_terms:
            return 'gender'
    return 'other'

# --- Build standardised dataframe ---
hx_clean = pd.DataFrame()
hx_clean['text'] = hx_train['post_tokens'].apply(lambda toks: clean_text(' '.join(toks)))
hx_clean['hate_label'] = hx_train['annotators'].apply(hx_majority_binary)
hx_clean['category'] = hx_train['annotators'].apply(hx_category)
hx_clean['dataset'] = 'HateXplain'

# --- Summary ---
print("HateXplain standardised:")
print(f"  Total posts: {len(hx_clean):,}")
print(f"  Hate (1): {(hx_clean['hate_label']==1).sum():,}")
print(f"  Non-hate (0): {(hx_clean['hate_label']==0).sum():,}")
print("\n  Category distribution:")
print(hx_clean['category'].value_counts())
print("\nFirst 3 rows:")
print(hx_clean.head(3))


Loading HateXplain...


Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/15383 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1922 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1924 [00:00<?, ? examples/s]

Loaded 15,383 training posts

HateXplain standardised:
  Total posts: 15,383
  Hate (1): 4,748
  Non-hate (0): 10,635

  Category distribution:
category
race        5819
other       5481
religion    2439
gender      1644
Name: count, dtype: int64

First 3 rows:
                                                                              text  \
0  u really think i would not have been raped by feral hindu or muslim back in ...   
1  the uk has threatened to return radioactive waste to the eu if an agreement ...   
2  if english is not imposition then hindi is also not imposition shut up chuti...   

   hate_label  category     dataset  
0           0  religion  HateXplain  
1           0      race  HateXplain  
2           0  religion  HateXplain  


In [ ]:
# ============================================================
# ETHOS - Load and harmonise to binary + category labels
# ============================================================
# ETHOS multilabel version provides separate category columns.
# The binary hate label is derived from the violence/hate columns;
# category columns (gender, race, religion) indicate the target.
# ============================================================

print("Loading ETHOS (multilabel)...")
ethos = load_dataset("iamollas/ethos", "multilabel")
ethos_df = pd.DataFrame(ethos['train'])
print(f"Loaded {len(ethos_df):,} posts\n")
print("Available columns:")
print(ethos_df.columns.tolist())
print("\nFirst row:")
print(ethos_df.iloc[0])

Loading ETHOS (multilabel)...


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/433 [00:00<?, ? examples/s]

Loaded 433 posts

Available columns:
['text', 'violence', 'directed_vs_generalized', 'gender', 'race', 'national_origin', 'disability', 'religion', 'sexual_orientation']

First row:
text                       You should know women's sports are a joke
violence                                                           0
directed_vs_generalized                                            0
gender                                                             1
race                                                               0
national_origin                                                    0
disability                                                         0
religion                                                           0
sexual_orientation                                                 0
Name: 0, dtype: object


In [ ]:
# ============================================================
# ETHOS harmonisation
# ============================================================
# Note: the multilabel ETHOS contains hate posts only, categorised
# by target dimension. All posts here are treated as hate (1).
# ETHOS therefore contributes to category-level hate analysis but,
# lacking categorised non-hate posts, is not used for false
# positive rate analysis (handled in methodology discussion).
# ============================================================

def ethos_category(row):
    # priority order: race, religion, gender (matching HateXplain)
    if row['race'] >= 0.5:
        return 'race'
    if row['religion'] >= 0.5:
        return 'religion'
    if row['gender'] >= 0.5:
        return 'gender'
    return 'other'

ethos_clean = pd.DataFrame()
ethos_clean['text'] = ethos_df['text'].apply(clean_text)
ethos_clean['hate_label'] = 1   # all multilabel ETHOS posts are hate
ethos_clean['category'] = ethos_df.apply(ethos_category, axis=1)
ethos_clean['dataset'] = 'ETHOS'

print("ETHOS standardised:")
print(f"  Total posts: {len(ethos_clean):,}")
print(f"  All labelled hate (1): {(ethos_clean['hate_label']==1).sum():,}")
print("\n  Category distribution:")
print(ethos_clean['category'].value_counts())
print("\nFirst 3 rows:")
print(ethos_clean.head(3))


ETHOS standardised:
  Total posts: 433
  All labelled hate (1): 433

  Category distribution:
category
other       195
gender       81
religion     81
race         76
Name: count, dtype: int64

First 3 rows:
                                                                              text  \
0                                        you should know women's sports are a joke   
1                                  you look like sloth with deeper down’s syndrome   
2  you look like russian and speak like indian. both are disgusting go kill you...   

   hate_label category dataset  
0           1   gender   ETHOS  
1           1    other   ETHOS  
2           1    other   ETHOS  


In [ ]:
# ============================================================
# MEASURING HATE SPEECH - Load and inspect
# ============================================================

print("Loading Measuring Hate Speech corpus...")
print("(Large dataset, may take a minute)\n")
mhs = load_dataset("ucberkeley-dlab/measuring-hate-speech")
mhs_df = pd.DataFrame(mhs['train'])
print(f"Loaded {len(mhs_df):,} annotations\n")

# Show the hate score column and the target category columns we need
print("Hate speech score column sample:")
print(mhs_df['hate_speech_score'].describe())

print("\nRace target columns:")
print([c for c in mhs_df.columns if c.startswith('target_race')])

print("\nReligion target columns:")
print([c for c in mhs_df.columns if c.startswith('target_religion')])

print("\nGender target columns:")
print([c for c in mhs_df.columns if c.startswith('target_gender')])


Loading Measuring Hate Speech corpus...
(Large dataset, may take a minute)



Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/135556 [00:00<?, ? examples/s]

Loaded 135,556 annotations

Hate speech score column sample:
count    135556.000000
mean         -0.567428
std           2.380003
min          -8.340000
25%          -2.330000
50%          -0.340000
75%           1.410000
max           6.300000
Name: hate_speech_score, dtype: float64

Race target columns:
['target_race_asian', 'target_race_black', 'target_race_latinx', 'target_race_middle_eastern', 'target_race_native_american', 'target_race_pacific_islander', 'target_race_white', 'target_race_other', 'target_race']

Religion target columns:
['target_religion_atheist', 'target_religion_buddhist', 'target_religion_christian', 'target_religion_hindu', 'target_religion_jewish', 'target_religion_mormon', 'target_religion_muslim', 'target_religion_other', 'target_religion']

Gender target columns:
['target_gender_men', 'target_gender_non_binary', 'target_gender_transgender_men', 'target_gender_transgender_unspecified', 'target_gender_transgender_women', 'target_gender_women', 'target_gender

In [ ]:
# ============================================================
# MEASURING HATE SPEECH - Harmonise to binary + category labels
# ============================================================
# This dataset is annotation-level (multiple annotators per comment).
# We aggregate to one row per comment for comparability with the
# post-level structure of HateXplain and ETHOS.
#
# Binary label: hate_speech_score > 0.5 -> hate (1), else non-hate (0)
#   (0.5 is the conventional threshold in the dataset documentation)
# Category: uses the aggregate target_race / target_religion /
#   target_gender columns, which capture all sub-targets including
#   non-binary and transgender gender targets.
# ============================================================

# --- Aggregate to one row per unique comment ---
agg = mhs_df.groupby('comment_id').agg({
    'text': 'first',
    'hate_speech_score': 'mean',
    'target_race': 'mean',
    'target_religion': 'mean',
    'target_gender': 'mean'
}).reset_index()

print(f"Aggregated to {len(agg):,} unique comments "
      f"(from {len(mhs_df):,} annotations)\n")

# --- Binary hate label from continuous score ---
agg['hate_label'] = (agg['hate_speech_score'] > 0.5).astype(int)

# --- Category assignment, priority order race > religion > gender ---
def mhs_category(row):
    if row['target_race'] >= 0.5:
        return 'race'
    if row['target_religion'] >= 0.5:
        return 'religion'
    if row['target_gender'] >= 0.5:
        return 'gender'
    return 'other'

agg['category'] = agg.apply(mhs_category, axis=1)

# --- Build standardised dataframe ---
mhs_clean = pd.DataFrame()
mhs_clean['text'] = agg['text'].apply(clean_text)
mhs_clean['hate_label'] = agg['hate_label']
mhs_clean['category'] = agg['category']
mhs_clean['dataset'] = 'MeasuringHateSpeech'

print("Measuring Hate Speech standardised:")
print(f"  Total comments: {len(mhs_clean):,}")
print(f"  Hate (1): {(mhs_clean['hate_label']==1).sum():,}")
print(f"  Non-hate (0): {(mhs_clean['hate_label']==0).sum():,}")
print("\n  Category distribution:")
print(mhs_clean['category'].value_counts())
print("\nFirst 3 rows:")
print(mhs_clean.head(3))

Aggregated to 39,565 unique comments (from 135,556 annotations)

Measuring Hate Speech standardised:
  Total comments: 39,565
  Hate (1): 10,354
  Non-hate (0): 29,211

  Category distribution:
category
race        12635
gender      12386
other        9453
religion     5091
Name: count, dtype: int64

First 3 rows:
                                                                              text  \
0                                                          white people are trash.   
1                 you've caught the big gay, it's known to be airborne via dildos.   
2  i'm not saying you said that, i'm just saying that if you find a slur offens...   

   hate_label category              dataset  
0           0     race  MeasuringHateSpeech  
1           0    other  MeasuringHateSpeech  
2           0     race  MeasuringHateSpeech  


In [ ]:
# ============================================================
# COMBINE AND SAVE - completing Stage 1
# ============================================================
# Combine the three standardised datasets into one master file
# and also save each individually. These clean CSVs are the input
# to the modelling notebook (Stage 2 onward), which avoids the
# library version conflict between dataset loading and the
# transformer models.
# ============================================================

# Combine into one master dataframe
master = pd.concat([hx_clean, ethos_clean, mhs_clean], ignore_index=True)

# Remove any empty text rows that cleaning may have produced
master = master[master['text'].str.strip() != ''].reset_index(drop=True)

print("COMBINED MASTER DATASET")
print("=" * 50)
print(f"Total posts: {len(master):,}")
print("\nBy dataset:")
print(master['dataset'].value_counts())
print("\nBy hate label:")
print(master['hate_label'].value_counts())
print("\nBy category:")
print(master['category'].value_counts())
print("\nHate label by category (the fairness analysis foundation):")
print(pd.crosstab(master['category'], master['hate_label']))

# --- Save the clean files ---
master.to_csv('master_clean.csv', index=False)
hx_clean.to_csv('hatexplain_clean.csv', index=False)
ethos_clean.to_csv('ethos_clean.csv', index=False)
mhs_clean.to_csv('measuring_hate_speech_clean.csv', index=False)

print("\n" + "=" * 50)
print("Saved: master_clean.csv, hatexplain_clean.csv,")
print("ethos_clean.csv, measuring_hate_speech_clean.csv")
print("Stage 1 (preprocessing) complete.")


COMBINED MASTER DATASET
Total posts: 55,381

By dataset:
dataset
MeasuringHateSpeech    39565
HateXplain             15383
ETHOS                    433
Name: count, dtype: int64

By hate label:
hate_label
0    39846
1    15535
Name: count, dtype: int64

By category:
category
race        18530
other       15129
gender      14111
religion     7611
Name: count, dtype: int64

Hate label by category (the fairness analysis foundation):
hate_label      0     1
category               
gender      10155  3956
other       12197  2932
race        12197  6333
religion     5297  2314

Saved: master_clean.csv, hatexplain_clean.csv,
ethos_clean.csv, measuring_hate_speech_clean.csv
Stage 1 (preprocessing) complete.
